# Image processing notebook: From overlap corrected to transmission 

### 00 - Introduction
This notebook demonstrates the use of Jupyter for a Time-of-Flight image processing task. The data corresponds to ToF neutron imaging of coin cells filled with different organic compounds and electrolytes.

*Note: This notebook was written and tested on Windows 10. Running on Mac or Linux machines may need adjustments, e.g. in the path specificationsApplied processing*

### Applied processing
The following processing parts from the averaged pulses, which were already overlap corrected:

- filtering
- Pulse averaging and separtion per experiments
- identification and weighting of OBs for each experiment
- scrubbing correction
- intensity correction
- transmission image generation


### Important Considerations
This notebook starts from the point **after** the overlap correction presented in the image below. This step before (done in a separate notebook) format its destionation directory to be taken by this notebook to process the images correctly.

Transmission Image:

\begin{equation}
T_{img} \rightarrow \frac{I}{I_{0}}=\frac{\frac{\bar{Img}}{OB_{weight}} - SBKG_{img}}{\frac{\bar{Ref}}{OB_{weight}} - SBKG_{ref}}
\end{equation}

## 01 Initial settings
Import all the required libraries

In [1]:
import sys
sys.path.append(r'..\framework')
sys.path.append(r'..\proc_functions')
from proc_functions import *
from img_proc_functions import *
%matplotlib inline

### Select directories
Select the source directory. This directory is where the images **after** the overlap correction were saved.
Select the destination directory. Here is where the transmission images are going to be saved.

In [2]:
# %load select_directory('src_dir')
src_dir = r"J:\900 Varia\2021\000_tony_data\03_Processed_step_by_step\00_Overlap_correction\exp1XX"

In [3]:
# %load select_directory('dst_dir')
dst_dir = r"J:\900 Varia\2021\000_tony_data\03_Processed_step_by_step\00_new_framweork_results\exp1XX"

### Select working folders
Once the directories are loaded, you can start to do a selection of the folders you want to process. <br>
The next function loads the folders availableas a visual aid. However, you can avoid this step just by looking at the source folder in the windows explorer.

In [4]:
test_dict = prep_stack_dict(src_dir)
for key in test_dict.keys():
    print(key)

00_ob
01_so_ref
02_exp102_00
02_exp102_01
02_exp102_02
02_exp102_03
02_exp102_04
03_ob_end


For this test we will select some folders that we want to process as '`proc_folder`' and our reference as '`ref_folder`'.

* _note: `proc-folder` can take several strings as value, for that reason is a list of strings. On the other hand, `ref_folder` is always one, for that reason it is just a string._

In [5]:
proc_folder = ["02_exp102_00"]

proc_ref_folder = ['01_so_ref']

## 02 choose sequence for filters and pre-processing

In [6]:
seq = [crop_img, outlier_removal]
seq = list(sequence_separator(seq, lambda i, j: i.__name__ == "stack_avg"))
seq = list(filter(None, seq))

## 03 give the parameters needed for the pre-processing

In [7]:
param_dict = dict (roi_crop = [1, 356, 509, 129], threshold = 0)

## 04 keep the folders needed and read the images

In [8]:
test_dict = prep_stack_dict (src_dir)
weights = weighting_func (src_dir)

keep_folder = proc_folder 
keep_key_weights (test_dict, weights, keep_folder)

# we slice just for checking purposes
test_dict = slice_vals_dict (test_dict, start_slice = 30, end_slice = 50)

get_imgs_dict(test_dict)

Reading Images: 100%|████████████████████████████| 3/3 [00:04<00:00,  1.36s/it]


## 05 do the pre-processing for the experiment

In [9]:
for idx in range (len(seq)):
    func_names = str([func.__name__ for func in seq[idx]])
    if seq[idx] == []:
        seq[idx].pop()
    #print(func_names)
    if 'stack_avg' in func_names:
        test_dict = exec_avg_dict(test_dict, sequence = seq[idx], count_time = False, **param_dict)
    else:
        test_dict = exec_proc_dict(test_dict, sequence = seq[idx] , count_time = False, **param_dict)

Processing : 100%|███████████████████████████████| 6/6 [00:29<00:00,  4.92s/it]


## 06 milestone (checking headers and showing an image

In [ ]:
#list_imgs = [[np.nanmean(item[0], axis=0) for item in sub_val] for sub_val in test_dict[proc_folder[0]]]
#test_img = np.nanmean(list_imgs, axis = 0)
list_imgs = avg_tof_imgs_dict (test_dict, output_type = 'tof')
test_img = 

len(list_imgs)

In [ ]:
show_img (test_img)
show_img (test_img, keep_fig =True)


hdr = test_dict[[proc_folder[0]][0][3][1]]
img = (test_img[0], hdr)

## 07 Perform Scrubbing Correction

In [ ]:
test_dict = scrubbing_corr_dict (test_dict, weights)

## 08 Perform Image registration

In [ ]:
reg_img = get_img(src_dir + '/reg_img_LE.fits')

In [ ]:
reg_rois = [([25, 19, 25, 90], 'v'), ([435, 19, 25, 90], 'v')]
show_img_rois(img[0], dr = [(reg_rois, 'yellow')])

In [ ]:
disp = float(0.03333)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])
img_reg_corr, M = img_registration (img, reg_img, rois_list = reg_rois, dof=['ty'], M=M)

In [ ]:
plt.rcParams['figure.figsize'] = [15, 8]
show_img(img_reg_corr[0]/reg_img[0], cmap = 'gray')
print(M)

In [ ]:
ref_mask_img = get_img(src_dir + '/bb_mask.fits')
img_mask_img = get_img(src_dir + '/bb_mask.fits')
reg_img = get_img(src_dir + '/reg_img_LE.fits')

reg_rois =  [([432, 19, 27, 90], 'v'),([23, 19, 27, 90], 'v')]

test_dict = img_registration_dict (test_dict, ref = reg_img, rois_list = reg_rois, dof=['ty'], M=M)

## 09 SBKG correction

In [ ]:
test_dict = SBKG_correction_dict (test_dict, img_mask_img)

## 10 TFC correction

In [ ]:
# %load select_rois(img, list_rois = ['nca'])
nca = [404, 74, 51, 10]

### Do same process for the ref

In [ ]:
ref_dict = prep_stack_dict (src_dir)
weights = weighting_func (src_dir)

keep_folder = proc_ref_folder 
keep_key_weights (ref_dict, weights, keep_folder)

# we slice just for checking purposes
ref_dict = slice_vals_dict (ref_dict, start_slice = 15, end_slice = 50)

get_imgs_dict(ref_dict)

for idx in range (len(seq)):
    func_names = str([func.__name__ for func in seq[idx]])
    if seq[idx] == []:
        seq[idx].pop()
    #print(func_names)
    if 'stack_avg' in func_names:
        ref_dict = exec_avg_dict(ref_dict, sequence = seq[idx], count_time = False, **param_dict)
    else:
        ref_dict = exec_proc_dict(ref_dict, sequence = seq[idx] , count_time = False, **param_dict)

ref_dict = scrubbing_corr_dict (ref_dict, weights)
ref_dict = img_registration_dict (ref_dict, ref = reg_img, rois_list = reg_rois, dof=['ty'], M=M)
ref_dict = SBKG_correction_dict (ref_dict, ref_mask_img)

In [ ]:
ref_avg_img = avg_tof_imgs_dict (ref_dict, output_type = 'img')

test_dict = avg_TFC_correction_dict (test_dict, ref_avg_img, nca = nca)

## 11 Save results

In [ ]:
param_dict.update(start_img_numb = 15)
save_dict (test_dict, dst_dir, img_type = 'transmission_img')

## 07 Full image processing
After all tests are correct, you can process and do the full image process for all the folders that you want with the captured parameters.

The result of using the next function is that all the transmission images generated with it are saved with a .fits extension in your destination folder and HE and LE sections will be saved in another 2 folders.

__Note:__ If you are sure fo the process, you can leave `proc_folder` empty `proc_folder = []`,the program will process all folders included in the source directory.